In [ ]:
# ==========================================
# CUSTOMER RETENTION + CHURN PREDICTION
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# ------------------------------------------
# Step 1: Load Dataset (NO PATH ISSUE)
# ------------------------------------------
try:
    df = pd.read_csv('C:/Users/kousthubha/Downloads/customer.csv')  # keep file in same folder
    print("✅ Loaded customer.csv")
except:
    print("⚠️ customer.csv not found → using sample data")

    # Sample dataset (fallback)
    data = {
        'CustomerID': np.repeat(np.arange(1, 51), 5),
        'InvoiceNo': np.random.randint(1000, 2000, 250),
        'Quantity': np.random.randint(1, 5, 250),
        'UnitPrice': np.random.uniform(10, 100, 250),
        'InvoiceDate': pd.date_range(start='2023-01-01', periods=250, freq='D')
    }
    df = pd.DataFrame(data)

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# ------------------------------------------
# Step 2: RFM ANALYSIS
# ------------------------------------------
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalAmount': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.reset_index(inplace=True)

# ------------------------------------------
# Step 3: Create Churn Label (BALANCED)
# ------------------------------------------
threshold = rfm['Recency'].median()

rfm['Churn'] = np.where(rfm['Recency'] > threshold, 'Churned', 'Active')
rfm['Churn_Label'] = rfm['Churn'].map({'Active': 0, 'Churned': 1})

print("\nChurn Distribution:")
print(rfm['Churn'].value_counts())

# ------------------------------------------
# Step 4: Visualization
# ------------------------------------------
plt.figure()
sns.countplot(x='Churn', data=rfm)
plt.title("Churn Distribution")
plt.savefig("churn.png")
plt.show()

# ------------------------------------------
# Step 5: ML Model
# ------------------------------------------
X = rfm[['Recency', 'Frequency', 'Monetary']]
y = rfm['Churn_Label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.5,
    random_state=42,
    stratify=y   

)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("\n✅ Model Trained Successfully")

# ------------------------------------------
# Step 6: ROC Curve
# ------------------------------------------
y_probs = model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_probs)
auc_score = roc_auc_score(y_test, y_probs)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig("roc_curve.png")
plt.show()

# ------------------------------------------
# Step 7: Confusion Matrix
# ------------------------------------------
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

disp.plot()
plt.title("Confusion Matrix")
plt.savefig("confusion_matrix.png")
plt.show()

# ------------------------------------------
# Step 8: Save Output
# ------------------------------------------
rfm.to_csv("rfm_output.csv", index=False)

print(f"\n✅ ROC-AUC Score: {auc_score:.2f}")
print("\n🎉 Project Completed Successfully!")